In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ankush87/corpus/corpus


### IMPORTS

In [2]:
import re
import nltk
import random

In [3]:
with open("/kaggle/input/datasets/ankush87/corpus/corpus","r") as f:
    data = f.read()

print("Data type",type(data))
print("Length of the data", len(data))

print("First 100 characters", data[0:100])


Data type <class 'str'>
Length of the data 4507025
First 100 characters __label__2 Stuning even for the non-gamer: This sound track was beautiful! It paints the senery in y


In [4]:
pattern = r"__label__[0-9]\s*"
cleaned_lines = [re.sub(pattern,"",line) for line in data.splitlines()]
cleaned_data = "\n".join(cleaned_lines)

print("First 100 characters of cleaned data", cleaned_data[0:100])

First 100 characters of cleaned data Stuning even for the non-gamer: This sound track was beautiful! It paints the senery in your mind so


### PRE PROCESSING


1. Split data into sentences using "\n" as the delimiter.
2. Split each sentence into tokens
3. Assign sentences into train or test sets.
4. Find tokens that appear at least N times in the training data.
5. Replace tokens that appear less than N times by "\<unk>"


#### Spliting and Tokenizing the sentence

In [5]:
def split_to_sentence(text):

    sentences = text.split("\n")
    return sentences

def sentence_to_token(sentences):
    n = len(sentences)
    tokenized_sentences = []

    for s in sentences:
        s = s.lower()
        tokens = nltk.word_tokenize(s)
        tokenized_sentences.append(tokens)

    return tokenized_sentences

def get_tokenized_data(data):

    sentences = split_to_sentence(data)
    tokens = sentence_to_token(sentences)

    return tokens

test_txt = "Hello world\n This is exciting\n It is friday!\n This is cat"
test_tokens = get_tokenized_data(test_txt)
print("Test: ",test_tokens)

Test:  [['hello', 'world'], ['this', 'is', 'exciting'], ['it', 'is', 'friday', '!'], ['this', 'is', 'cat']]


#### Train and Test split

In [6]:
tokenized_data = get_tokenized_data(cleaned_data)
random.seed(33)
random.shuffle(tokenized_data)

# Training data:0.8 Test data:0.2
train_size = int(len(tokenized_data) * 0.8)
train_data = tokenized_data[0:train_size]
test_data = tokenized_data[train_size:]

print("{} data are split into {} train and {} test set".format(
    len(tokenized_data), len(train_data), len(test_data)))

print("First training sample:")
print(train_data[0])
      
print("First test sample")
print(test_data[0])


10000 data are split into 8000 train and 2000 test set
First training sample:
['better', 'than', 'mac', "'s", 'charger', ':', 'like', 'some', 'other', 'reviewers', ',', 'the', 'product', 'i', 'received', 'did', 'not', 'look', 'exactly', 'like', 'the', 'one', 'pictured', '.', 'that', "'s", 'irritating', '.', 'but', ',', 'it', 'functions', 'perfectly', ',', 'which', 'is', 'more', 'than', 'i', 'can', 'say', 'for', 'the', 'expensive', 'mac', 'chargers.mac', 'stores', 'will', 'tell', 'you', 'it', "'s", 'a', '``', 'myth', "''", 'that', 'their', 'chargers', 'break', '.', 'um', '...', 'why', 'is', 'it', 'then', 'that', 'me', ',', 'my', 'brother', ',', 'three', 'friends', 'and', 'several', 'acquaintances', 'have', 'had', 'this', 'problem', '?']
First test sample
['awesome', 'movie', 'a', 'must', 'see', ':', 'i', 'loved', 'this', 'movie', 'plain', 'and', 'simple', 'it', 'is', 'a', 'must', 'see', '.', 'zoey', 'saldana', 'has', 'a', 'huge', 'impact', 'in', 'this', 'film', '.', 'i', 'have', 'multip

In [7]:
def get_word_counts(tokenized_sentences):

    #dictionary to store the counts
    word_counts = {}
    
    for sentence in tokenized_sentences:

        for token in sentence:
            if token in word_counts.keys():
                word_counts[token] += 1
            else:
                word_counts[token] = 1

    return word_counts

print("Test:", get_word_counts(test_tokens))
    

Test: {'hello': 1, 'world': 1, 'this': 2, 'is': 3, 'exciting': 1, 'it': 1, 'friday': 1, '!': 1, 'cat': 1}


#### Handling Out of vocabulary words

A list of the most frequent words in the training set, called the closed vocabulary will be created.
We will convert all the other words that are not part of this  closed vocabulary to the token 'unk'.

In [8]:
def get_word_n_frequency(tokenized_sentences,count_threshold):
    """
    count_threshold: minimum number of occurrences for a word 
    to be in the closed vocabulary.
    """

    closed_vocab = []
    word_counts = get_word_counts(tokenized_sentences)

    for word,cnt in word_counts.items():
        if cnt >= count_threshold:
            closed_vocab.append(word)

    return closed_vocab

print("Test text:\n", test_txt)
print("\n closed vocab for the test:", get_word_n_frequency(test_tokens,2))
    

Test text:
 Hello world
 This is exciting
 It is friday!
 This is cat

 closed vocab for the test: ['this', 'is']


In [9]:
def oov_to_unk(tokenized_sentences, vocab, unknown_token="<unk>"):

    vocab = set(vocab)
    
    replaced_tokenized_sentences = []
    
    for sentence in tokenized_sentences:
        
        replaced_sentence = []
        
        for token in sentence: 
            if token in vocab: 
                replaced_sentence.append(token)
            else:
                replaced_sentence.append(unknown_token)

        replaced_tokenized_sentences.append(replaced_sentence)
        
    return replaced_tokenized_sentences

In [10]:
test_tokens = [['hello', 'world'], ['this', 'is', 'exciting'], ['it', 'is', 'friday', '!'], ['this', 'is', 'cat']]
test_vocab = get_word_n_frequency(test_tokens,2) #returns: ["this","is"]
replaced_sentence = oov_to_unk(test_tokens, test_vocab, unknown_token="<unk>")
print("Original sentence:", test_tokens)
print("After replacing less frequent words with <unk>: ",replaced_sentence)

Original sentence: [['hello', 'world'], ['this', 'is', 'exciting'], ['it', 'is', 'friday', '!'], ['this', 'is', 'cat']]
After replacing less frequent words with <unk>:  [['<unk>', '<unk>'], ['this', 'is', '<unk>'], ['<unk>', 'is', '<unk>', '<unk>'], ['this', 'is', '<unk>']]


In [11]:
def preprocess_data(train_data, test_data, count_threshold):

    # closed vocab using train data
    vocabulary = get_word_n_frequency(train_data,count_threshold)

    #replace less frequent words with "unk"
    train_data_replaced = oov_to_unk(train_data,vocabulary)
    test_data_replaced = oov_to_unk(test_data,vocabulary)

    return train_data_replaced, test_data_replaced, vocabulary

In [12]:
minimum_freq = 2
train_data_processed, test_data_processed, vocabulary = preprocess_data(train_data, 
                                                                        test_data, 
                                                                        minimum_freq)
print("First preprocessed training sample:")
print(train_data_processed[0])
print()
print("First preprocessed test sample:")
print(test_data_processed[0])
print()
print("First 10 vocabulary:")
print(vocabulary[0:10])
print()
print("Size of vocabulary:", len(vocabulary))

First preprocessed training sample:
['better', 'than', 'mac', "'s", 'charger', ':', 'like', 'some', 'other', 'reviewers', ',', 'the', 'product', 'i', 'received', 'did', 'not', 'look', 'exactly', 'like', 'the', 'one', 'pictured', '.', 'that', "'s", 'irritating', '.', 'but', ',', 'it', 'functions', 'perfectly', ',', 'which', 'is', 'more', 'than', 'i', 'can', 'say', 'for', 'the', 'expensive', 'mac', '<unk>', 'stores', 'will', 'tell', 'you', 'it', "'s", 'a', '``', 'myth', "''", 'that', 'their', 'chargers', 'break', '.', 'um', '...', 'why', 'is', 'it', 'then', 'that', 'me', ',', 'my', 'brother', ',', 'three', 'friends', 'and', 'several', 'acquaintances', 'have', 'had', 'this', 'problem', '?']

First preprocessed test sample:
['awesome', 'movie', 'a', 'must', 'see', ':', 'i', 'loved', 'this', 'movie', 'plain', 'and', 'simple', 'it', 'is', 'a', 'must', 'see', '.', '<unk>', '<unk>', 'has', 'a', 'huge', 'impact', 'in', 'this', 'film', '.', 'i', 'have', 'multiple', 'copies', 'of', 'this', 'film'

### N-GRAM BASED LANGUAGE MODEL

##### Count n-grams in the data

In [13]:
def count_n_grams(data, n, start_token='<s>', end_token = '<e>'):
    n_grams = {}
    for sentence in data: # complete this line
        sentence = [start_token] * n+ sentence + [end_token]
        sentence = tuple(sentence)
        m = len(sentence) if n==1 else len(sentence)-1
        for i in range(m): # complete this line
            n_gram = sentence[i:i+n]
            if n_gram in n_grams.keys(): # complete this line
                n_grams[n_gram] += 1
            else:
                n_grams[n_gram] = 1
    return n_grams

In [14]:
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
print("Uni-gram:")
print(count_n_grams(sentences, 1))
print("Bi-gram:")
print(count_n_grams(sentences, 2))

Uni-gram:
{('<s>',): 2, ('i',): 1, ('like',): 2, ('a',): 2, ('cat',): 2, ('<e>',): 2, ('this',): 1, ('dog',): 1, ('is',): 1}
Bi-gram:
{('<s>', '<s>'): 2, ('<s>', 'i'): 1, ('i', 'like'): 1, ('like', 'a'): 2, ('a', 'cat'): 2, ('cat', '<e>'): 2, ('<s>', 'this'): 1, ('this', 'dog'): 1, ('dog', 'is'): 1, ('is', 'like'): 1}


##### Estimate the probability of a word given the prior 'n' words

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n)}{C(w_{t-1}\dots w_{t-n})} \tag{2} $$


In [15]:
def estimate_probability(word, previous_n_gram, 
                         n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    """
        n_gram_counts: Dictionary of counts of n-grams
        n_plus1_gram_counts: Dictionary of counts of (n+1)-grams
        k: positive constant, smoothing parameter
    
    Returns:
        A probability
    """
    previous_n_gram = tuple(previous_n_gram) 
    
    previous_n_gram_count = n_gram_counts[previous_n_gram] if previous_n_gram in n_gram_counts  else 0
    denominator = previous_n_gram_count + k * vocabulary_size
    n_plus1_gram = previous_n_gram + (word,)
    
    n_plus1_gram_count = n_plus1_gram_counts[n_plus1_gram] if n_plus1_gram in n_plus1_gram_counts  else 0
    numerator = n_plus1_gram_count + k
    
    probability = numerator / denominator
    return probability
    

In [16]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
tmp_prob = estimate_probability("cat", "a", unigram_counts, bigram_counts, len(unique_words), k=1)
print(f"The estimated probability of word 'cat' given the previous n-gram 'a' is: {tmp_prob:.4f}")

The estimated probability of word 'cat' given the previous n-gram 'a' is: 0.3333


In [17]:
def estimate_probabilities(previous_n_gram, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0):

    previous_n_gram = tuple(previous_n_gram)
    vocabulary = vocabulary + ["<e>", "<unk>"]
    vocabulary_size = len(vocabulary)
    
    probabilities = {}
    for word in vocabulary:
        probability = estimate_probability(word, previous_n_gram, 
                                           n_gram_counts, n_plus1_gram_counts, 
                                           vocabulary_size, k=k)
        probabilities[word] = probability

    return probabilities

In [18]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
estimate_probabilities("a", unigram_counts, bigram_counts, unique_words, k=1)

{'i': 0.09090909090909091,
 'dog': 0.09090909090909091,
 'a': 0.09090909090909091,
 'cat': 0.2727272727272727,
 'like': 0.09090909090909091,
 'this': 0.09090909090909091,
 'is': 0.09090909090909091,
 '<e>': 0.09090909090909091,
 '<unk>': 0.09090909090909091}

##### Count and Probability matrix

In [19]:
def make_count_matrix(n_plus1_gram_counts, vocabulary):
 
    vocabulary = vocabulary + ["<e>", "<unk>"]
    n_grams = []
    
    for n_plus1_gram in n_plus1_gram_counts.keys():
        n_gram = n_plus1_gram[0:-1]
        n_grams.append(n_gram)
    n_grams = list(set(n_grams))
    
    row_index = {n_gram:i for i, n_gram in enumerate(n_grams)}
    col_index = {word:j for j, word in enumerate(vocabulary)}
    
    nrow = len(n_grams)
    ncol = len(vocabulary)
    count_matrix = np.zeros((nrow, ncol))
    for n_plus1_gram, count in n_plus1_gram_counts.items():
        n_gram = n_plus1_gram[0:-1]
        word = n_plus1_gram[-1]
        if word not in vocabulary:
            continue
        i = row_index[n_gram]
        j = col_index[word]
        count_matrix[i, j] = count
    
    count_matrix = pd.DataFrame(count_matrix, index=n_grams, columns=vocabulary)
    return count_matrix

In [20]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)

print('bigram counts')
display(make_count_matrix(bigram_counts, unique_words))

bigram counts


,i,dog,a,cat,like,this,is,<e>,<unk>
"(i,)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
"(is,)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
"(cat,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
"(<s>,)",1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
"(dog,)",0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
"(like,)",0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
"(this,)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(a,)",0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0


In [21]:
print('\ntrigram counts')
trigram_counts = count_n_grams(sentences, 3)
display(make_count_matrix(trigram_counts, unique_words))


trigram counts


,i,dog,a,cat,like,this,is,<e>,<unk>
"(<s>, <s>)",1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
"(cat,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
"(<s>, this)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(is, like)",0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
"(a, cat)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
"(dog, is)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
"(<s>, i)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
"(i, like)",0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
"(like, a)",0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
"(this, dog)",0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [22]:
def make_probability_matrix(n_plus1_gram_counts, vocabulary, k):
    count_matrix = make_count_matrix(n_plus1_gram_counts, unique_words)
    count_matrix += k
    prob_matrix = count_matrix.div(count_matrix.sum(axis=1), axis=0)
    return prob_matrix

In [23]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)
print("bigram probabilities")
display(make_probability_matrix(bigram_counts, unique_words, k=1))

bigram probabilities


,i,dog,a,cat,like,this,is,<e>,<unk>
"(i,)",0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000
"(is,)",0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000
"(cat,)",0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.272727,0.090909
"(<s>,)",0.181818,0.090909,0.090909,0.090909,0.090909,0.181818,0.090909,0.090909,0.090909
"(dog,)",0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000
"(like,)",0.090909,0.090909,0.272727,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909
"(this,)",0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000
"(a,)",0.090909,0.090909,0.090909,0.272727,0.090909,0.090909,0.090909,0.090909,0.090909


### EVALUATION METRIC - Perplexity


$$ PP(W) =\sqrt[N]{ \prod_{t=n}^{N-1} \frac{1}{P(w_t | w_{t-n} \cdots w_{t-1})} } \tag{4.1}$$


In [24]:
# UNQ_C10 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: calculate_perplexity
def calculate_perplexity(sentence, n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):

    n = len(list(n_gram_counts.keys())[0]) 
    sentence = ["<s>"] * n + sentence + ["<e>"]
    sentence = tuple(sentence)
    N = len(sentence)
    product_pi = 1.0
    for t in range(n, N): # complete this line

        n_gram = sentence[t-n:t]
        word = sentence[t]
        probability = estimate_probability(word,n_gram, n_gram_counts, n_plus1_gram_counts, len(unique_words), k=1)
        product_pi *= 1 / probability

    # Nth root of the product
    perplexity = product_pi**(1/float(N))
    
    return perplexity

In [25]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)


perplexity_train1 = calculate_perplexity(sentences[0],
                                         unigram_counts, bigram_counts,
                                         len(unique_words), k=1.0)
print(f"Perplexity for first train sample: {perplexity_train1:.4f}")

test_sentence = ['i', 'like', 'a', 'dog']
perplexity_test = calculate_perplexity(test_sentence,
                                       unigram_counts, bigram_counts,
                                       len(unique_words), k=1.0)
print(f"Perplexity for test sample: {perplexity_test:.4f}")

Perplexity for first train sample: 2.8040
Perplexity for test sample: 3.9654


### TEXT GENERATION

In [26]:
def suggest_a_word(previous_tokens, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0, start_with=None):
    """
    Get suggestion for the next word
    
    Args:
        previous_tokens: The sentence you input where each token is a word. Must have length > n 
        n_gram_counts: Dictionary of counts of (n+1)-grams
        n_plus1_gram_counts: Dictionary of counts of (n+1)-grams
        vocabulary: List of words
        k: positive constant, smoothing parameter
        start_with: If not None, specifies the first few letters of the next word
        
    Returns:
        A tuple of 
          - string of the most likely next word
          - corresponding probability
    """
    
    # length of previous words
    n = len(list(n_gram_counts.keys())[0]) 
    
    # From the words that the user already typed
    # get the most recent 'n' words as the previous n-gram
    previous_n_gram = previous_tokens[-n:]

    # Estimate the probabilities that each word in the vocabulary
    # is the next word,
    # given the previous n-gram, the dictionary of n-gram counts,
    # the dictionary of n plus 1 gram counts, and the smoothing constant
    probabilities = estimate_probabilities(previous_n_gram,
                                           n_gram_counts, n_plus1_gram_counts,
                                           vocabulary, k=k)
    
    # Initialize suggested word to None
    # This will be set to the word with highest probability
    suggestion = None
    
    # Initialize the highest word probability to 0
    # this will be set to the highest probability 
    # of all words to be suggested
    max_prob = 0
    
    # For each word and its probability in the probabilities dictionary:
    for word, prob in probabilities.items(): # complete this line
        
        # If the optional start_with string is set
        if start_with != None: # complete this line

            # Check if the word starts with the letters in 'start_with'
            if not word.startswith(start_with): # complete this line

                #If so, don't consider this word (move onto the next word)
                continue # complete this line
        
        # Check if this word's probability
        # is greater than the current maximum probability
        if prob > max_prob: # complete this line
            
            # If so, save this word as the best suggestion (so far)
            suggestion = word
            
            # Save the new maximum probability
            max_prob = prob
    
    return suggestion, max_prob

In [27]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)

previous_tokens = ["i", "like"]
tmp_suggest1 = suggest_a_word(previous_tokens, unigram_counts, bigram_counts, unique_words, k=1.0)
print(f"The previous words are 'i like',\n\tand the suggested word is `{tmp_suggest1[0]}` with a probability of {tmp_suggest1[1]:.4f}")

print()
# test your code when setting the starts_with
tmp_starts_with = 'c'
tmp_suggest2 = suggest_a_word(previous_tokens, unigram_counts, bigram_counts, unique_words, k=1.0, start_with=tmp_starts_with)
print(f"The previous words are 'i like', the suggestion must start with `{tmp_starts_with}`\n\tand the suggested word is `{tmp_suggest2[0]}` with a probability of {tmp_suggest2[1]:.4f}")

The previous words are 'i like',
	and the suggested word is `a` with a probability of 0.2727

The previous words are 'i like', the suggestion must start with `c`
	and the suggested word is `cat` with a probability of 0.0909


In [28]:
def get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with=None):
    model_counts = len(n_gram_counts_list)
    suggestions = []
    for i in range(model_counts-1):
        n_gram_counts = n_gram_counts_list[i]
        n_plus1_gram_counts = n_gram_counts_list[i+1]
        
        suggestion = suggest_a_word(previous_tokens, n_gram_counts,
                                    n_plus1_gram_counts, vocabulary,
                                    k=k, start_with=start_with)
        suggestions.append(suggestion)
    return suggestions

In [29]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
trigram_counts = count_n_grams(sentences, 3)
quadgram_counts = count_n_grams(sentences, 4)
qintgram_counts = count_n_grams(sentences, 5)

n_gram_counts_list = [unigram_counts, bigram_counts, trigram_counts, quadgram_counts, qintgram_counts]
previous_tokens = ["i", "like"]
tmp_suggest3 = get_suggestions(previous_tokens, n_gram_counts_list, unique_words, k=1.0)

print(f"The previous words are 'i like', the suggestions are:")
display(tmp_suggest3)

The previous words are 'i like', the suggestions are:


[('a', 0.2727272727272727),
 ('a', 0.2),
 ('i', 0.1111111111111111),
 ('i', 0.1111111111111111)]

##### Training the model

In [30]:
n_gram_counts_list = []
for n in range(1, 6):
    print("Computing n-gram counts with n =", n, "...")
    n_model_counts = count_n_grams(train_data_processed, n)
    n_gram_counts_list.append(n_model_counts)

Computing n-gram counts with n = 1 ...
Computing n-gram counts with n = 2 ...
Computing n-gram counts with n = 3 ...
Computing n-gram counts with n = 4 ...
Computing n-gram counts with n = 5 ...


In [31]:

previous_tokens = ["hey", "how", "are", "you"]
tmp_suggest7 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest7)

The previous words are ['hey', 'how', 'are', 'you'], the suggestions are:


[('are', 0.020922720917900015),
 ('in', 0.0003945551390806865),
 ('supposed', 0.00013176966662274345),
 ('better', 6.588917440864466e-05)]

In [32]:
previous_tokens = ["i", "love", "to"]
tmp_suggest7 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest7)

The previous words are ['i', 'love', 'to'], the suggestions are:


[('the', 0.03763141519012216),
 ('see', 0.0005258314710135402),
 ('read', 0.0001975763962065332),
 ('better', 6.588917440864466e-05)]

In [33]:
previous_tokens = ["this", "is"]
tmp_suggest7 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest7)

The previous words are ['this', 'is'], the suggestions are:


[('a', 0.06286576613051639),
 ('a', 0.02780920621687608),
 ('<e>', 0.00013177834881728933),
 ('<e>', 0.00013177834881728933)]